In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from transformers import AutoTokenizer
from tqdm.auto import tqdm
import numpy as np
from sklearn.model_selection import train_test_split
import gc
import pandas as pd

In [ ]:
# --- Tokenizer & Constants ---
tokenizer = AutoTokenizer.from_pretrained("DeepChem/SmilesTokenizer_PubChem_1M")
VOCAB_SIZE = tokenizer.vocab_size
PAD_IDX = tokenizer.pad_token_id
ALL_SPECIAL = set(tokenizer.all_special_tokens)

print(f"Vocab size: {VOCAB_SIZE}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/672 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/420 [00:00<?, ?B/s]

Vocab size: 591


In [ ]:
class QM9SmilesDataset(Dataset):
    def __init__(self, csv_path):
        df = pd.read_csv(csv_path, usecols=['mol_id', 'smiles'])
        self.mol_ids = df['mol_id'].tolist()
        self.smiles = df['smiles'].tolist()

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        smiles = self.smiles[idx]
        # Tokenize the SMILES string
        # Assuming 'tokenizer' is globally available from the previous cell
        tokenized_output = tokenizer(
            smiles,
            padding='max_length',
            truncation=True,
            max_length=64, # Matches max_seq_len in model, adjust as needed
            return_tensors='pt'
        )

        return {
            'mol_id': self.mol_ids[idx],
            'input_ids': tokenized_output['input_ids'].squeeze(0), # Remove batch dim
            'attention_mask': tokenized_output['attention_mask'].squeeze(0)
        }

In [ ]:
# ----------------------------
# Helper: additive causal mask
# ----------------------------
def causal_attn_mask(L, device):
    """
    Returns an additive attention mask of shape (L, L) where positions
    in the future (j > i) are set to -inf, and allowed positions are 0.
    This is the format nn.MultiheadAttention expects for attn_mask.
    """
    # upper triangular w/ diagonal=1 -> future positions = 1
    mask = torch.triu(torch.ones((L, L), device=device, dtype=torch.bool), diagonal=1)
    # convert bool to float additive mask: 0 (allow) or -inf (mask)
    if mask.any():
        additive = torch.full((L, L), float("-inf"), device=device)
        additive = torch.where(mask, additive, torch.zeros_like(additive))
    else:
        additive = torch.zeros((L, L), device=device)
    return additive  # shape: (L, L)

# ----------------------------
# GPT Decoder Block
# ----------------------------
class GPTDecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=n_heads, dropout=dropout, batch_first=True
        )
        self.dropout1 = nn.Dropout(dropout)

        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, attn_mask=None, key_padding_mask=None):
        # x: (B, L, d_model)
        h = self.ln1(x)
        if attn_mask is not None:
            attn_mask = attn_mask.bool()

        if key_padding_mask is not None:
            key_padding_mask = key_padding_mask.bool()

        attn_out, _ = self.attn(h, h, h, attn_mask=attn_mask, key_padding_mask=key_padding_mask, need_weights=False)
        x = x + self.dropout1(attn_out)

        h2 = self.ln2(x)
        ff_out = self.ff(h2)
        x = x + self.dropout2(ff_out)
        return x


# ----------------------------
# GPT-style SMILES LM
# ----------------------------
class SMILESGPT(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model=256,
        n_layers=6,
        n_heads=8,
        d_ff=1024,
        max_seq_len=256,
        dropout=0.1,
        tie_word_embeddings=True,
    ):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.dropout = nn.Dropout(dropout)

        self.blocks = nn.ModuleList(
            [GPTDecoderBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)]
        )

        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        if tie_word_embeddings:
            # Weight tying: lm_head uses the same weight matrix as token_emb
            self.lm_head.weight = self.token_emb.weight

        self.max_seq_len = max_seq_len
        self.vocab_size = vocab_size

    def forward(self, idx, attention_mask=None):
        """
        idx: (B, L) token ids
        attention_mask: Optional key_padding_mask of shape (B, L) with True in positions to be masked (padding)
                        (Note: this is 'key_padding_mask' argument for MultiheadAttention)
        """
        B, L = idx.shape
        device = idx.device
        if L > self.max_seq_len:
            raise ValueError(f"Sequence length {L} > max_seq_len {self.max_seq_len}")

        # token + positional embeddings
        pos_ids = torch.arange(L, device=device).unsqueeze(0)  # (1, L)
        x = self.token_emb(idx) + self.pos_emb(pos_ids)  # broadcasting pos -> (B, L, d)
        x = self.dropout(x)

        # additive causal attn mask (L, L) with -inf for future
        attn_mask = causal_attn_mask(L, device)

        # key_padding_mask: expects shape (B, L) with True in positions to mask (padding)
        key_padding_mask = None
        if attention_mask is not None:
            # If user passes attention_mask as 1=attend, 0=pad style, convert to bool mask
            # We expect attention_mask where 1 indicates token, 0 indicates padding.
            # Convert to key_padding_mask where True means to be masked.
            key_padding_mask = attention_mask == 0

        for block in self.blocks:
            x = block(x, attn_mask=attn_mask, key_padding_mask=key_padding_mask)

        x = self.ln_f(x)
        logits = self.lm_head(x)  # (B, L, V)
        return logits


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# Data splits
full_dataset = QM9SmilesDataset('qm9.csv')
train_dataset, val_dataset = train_test_split(
    full_dataset, test_size=0.1, random_state=42
)

In [ ]:
model_dim = 128
n_layers = 3

In [ ]:
model = SMILESGPT(
        vocab_size=VOCAB_SIZE,
        d_model=model_dim,
        n_layers=n_layers,
        n_heads=4,
        d_ff=model_dim*4,
        max_seq_len=64,
        dropout=0.1
    ).to(device)

In [ ]:
model.load_state_dict(torch.load('smiles_gpt_best.pth'))

<All keys matched successfully>

In [ ]:
def generate(model, prompt, max_new_tokens=50, temperature=0.8, device="cuda"):
    model.eval()
    with torch.no_grad():
        # Tokenize the prompt
        input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(device)

        for _ in range(max_new_tokens):
            # If the sequence length exceeds max_seq_len, truncate it for the model input
            current_input_ids = input_ids if input_ids.size(1) <= model.max_seq_len else input_ids[:, -model.max_seq_len:]

            # Get predictions (logits)
            logits = model(current_input_ids)

            # Focus only on the last token's logits
            logits = logits[:, -1, :] / temperature

            # Apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1)

            # Sample from the distribution
            next_token_id = torch.multinomial(probs, num_samples=1)

            # Break if the model generates an end-of-sequence token (if applicable)
            # For SMILES, we might not have a specific EOS token, or it's handled implicitly.
            # For now, just append.

            # Append sampled token to the input
            input_ids = torch.cat([input_ids, next_token_id], dim=-1)

            # Stop if we generate a special token that indicates end of generation (e.g., PAD_IDX or a specific EOS if defined)
            if next_token_id.item() == PAD_IDX:
                break

        # Decode the generated sequence
        generated_smiles = tokenizer.decode(input_ids.squeeze(0), skip_special_tokens=True)
    return generated_smiles

In [ ]:
df = pd.read_csv('qm9.csv', usecols=['mol_id', 'smiles'])

In [ ]:
df.head()

,mol_id,smiles
0,gdb_1,C
1,gdb_2,N
2,gdb_3,O
3,gdb_4,C#C
4,gdb_5,C#N


In [ ]:
val_dataset[0]

{'mol_id': 'gdb_14175',
 'input_ids': tensor([12, 16, 16, 20, 17, 16, 18, 16, 21, 16, 19, 16, 16, 20, 21, 13,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0]),
 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])}

In [ ]:
val_dataset_short = val_dataset[:2]

In [ ]:
# generate samples

d_result = {
    'mol_id': [],
    'random_position': [],
    'initial_seed_smiles': [],
    'generate_smiles': [],
    'ground_truth_answer_smiles': []
}

model.eval()

for data_point in tqdm(val_dataset, desc="Generating samples"):
    mol_id = data_point['mol_id']
    input_ids = data_point['input_ids']
    attention_mask = data_point['attention_mask']

    # Get actual sequence length (excluding padding)
    seq_len = attention_mask.sum().item()

    # Skip if sequence is too short
    if seq_len < 2:
        continue

    # Random position from 1 to seq_len-1 (exclude first token and last position)
    random_pos = np.random.randint(1, seq_len)

    # Extract seed and ground truth
    seed_ids = input_ids[:random_pos]
    full_ids = input_ids[:seq_len]

    # Decode
    seed_smiles = tokenizer.decode(seed_ids, skip_special_tokens=True)
    ground_truth_smiles = tokenizer.decode(full_ids, skip_special_tokens=True)

    # Generate continuation
    generated_smiles = generate(
        model,
        seed_smiles,
        max_new_tokens=64-random_pos,
        temperature=0.8,
        device=device
    )

    # Store results
    d_result['mol_id'].append(mol_id)
    d_result['random_position'].append(random_pos)
    d_result['initial_seed_smiles'].append(seed_smiles)
    d_result['generate_smiles'].append(generated_smiles)
    d_result['ground_truth_answer_smiles'].append(ground_truth_smiles)

# Convert to DataFrame
results_df = pd.DataFrame(d_result)
print(f"Generated {len(results_df)} samples")
results_df.head()

Generating samples:   0%|          | 0/7075 [00:00<?, ?it/s]

Generated 7075 samples


,mol_id,random_position,initial_seed_smiles,generate_smiles,ground_truth_answer_smiles
0,gdb_14175,9,CC1(C)C2,CC1(C)C2CC1(O)C2,CC1(C)C2COCC12
1,gdb_60332,1,,C12CCC1C(=O)OC2,CC(O)CCC(=O)C=O
2,gdb_29215,11,c1c(c(cnc1,c1c(c(cnc1)F)N,c1c(c(cnc1=O)N)O
3,gdb_7256,9,CC(C)C(=,CC(C)C(=CN)NC=O,CC(C)C(=O)NC=O
4,gdb_17484,4,CC1,CC1CC2(O)CC12C,CC12CC3C1N3CC2


In [ ]:
results_df.to_csv('smiles-gpt-generated.csv')